[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/ia-empresarial/04-ia-producto.ipynb)

# IA en Producto: User Research, Priorización y A/B Testing

Herramientas para PMs: síntesis de entrevistas, RICE scoring, user stories y análisis de reviews.

In [ ]:
import anthropic
import json

client = anthropic.Anthropic()
print("Herramientas de producto con IA listas")

## 1. Síntesis de entrevistas de usuario

In [ ]:
# Transcripciones de entrevistas simuladas
ENTREVISTAS = [
    """Entrevista con Ana, Directora de RRHH:
    El mayor problema que tenemos es el onboarding de nuevos empleados. Tardamos 3 semanas en dar acceso
    a todos los sistemas. La herramienta actual es muy manual, hay que ir sistema por sistema.
    También echo de menos poder ver en un solo sitio quién tiene acceso a qué. Y los informes
    para auditoría son un desastre, los hacemos en Excel.""",

    """Entrevista con Carlos, IT Manager:
    Lo que más me molesta es que cuando alguien deja la empresa, tengo que acordarme de revocar
    acceso en 12 sistemas distintos. Ya hemos tenido incidentes de seguridad por esto.
    Necesitaría automatización y alertas. También los empleados siempre me piden acceso a cosas
    y no tengo forma fácil de aprobar o denegar con registro.""",

    """Entrevista con María, CFO:
    Estamos pagando por licencias de software que nadie usa. Necesito visibility de qué herramientas
    usa realmente cada equipo para optimizar el presupuesto. El año pasado gastamos 45.000€ en licencias,
    calculo que un 30% fueron desperdicio. También necesito saber el coste por empleado para los
    informes de junta.""",

    """Entrevista con Pedro, nuevo empleado (3 meses):
    El primer mes fue frustrante. No sabía a qué sistemas tenía acceso ni cómo pedirlo.
    Al final preguntaba a compañeros. La herramienta de solicitudes de acceso es muy poco intuitiva,
    los formularios son confusos. Tardé 2 semanas en conseguir acceso al repositorio de código."""
]

def sintetizar_entrevistas(transcripciones: list, pregunta_investigacion: str) -> dict:
    """Sintetiza múltiples entrevistas de usuario en insights accionables."""
    texto_entrevistas = "\n\n---\n".join(
        f"Entrevista {i+1}:\n{t}"
        for i, t in enumerate(transcripciones)
    )

    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=2000,
        system="Eres un UX researcher experto en síntesis de investigación cualitativa.",
        messages=[{
            "role": "user",
            "content": f"""Sintetiza estas {len(transcripciones)} entrevistas de usuario.

PREGUNTA DE INVESTIGACIÓN: {pregunta_investigacion}

{texto_entrevistas}

Devuelve JSON:
{{
  "temas_principales": [
    {{"tema": "nombre", "frecuencia": "3/4 personas",
      "citas": ["cita textual"], "insight": "qué significa para el producto"}}
  ],
  "necesidades_no_satisfechas": ["necesidad"],
  "oportunidades_producto": ["oportunidad"],
  "segmentos_usuarios": [{{"perfil": "nombre", "necesidades": ["..."]}}],
  "hipotesis_a_validar": ["hipótesis"]
}}"""
        }]
    )
    texto = resp.content[0].text
    if "```" in texto:
        texto = texto.split("```")[1].lstrip("json\n")
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        return {"sintesis": texto}


sintesis = sintetizar_entrevistas(
    ENTREVISTAS,
    "¿Cuáles son los mayores problemas con la gestión de accesos y licencias de software?"
)

print("SÍNTESIS DE ENTREVISTAS")
print("=" * 60)

print("\n📌 TEMAS PRINCIPALES:")
for tema in sintesis.get("temas_principales", []):
    print(f"\n  • {tema.get('tema')} ({tema.get('frecuencia')})")
    print(f"    Insight: {tema.get('insight', '')}")
    for cita in tema.get("citas", [])[:1]:
        print(f"    \"{cita[:100]}\"")

print("\n🎯 OPORTUNIDADES DE PRODUCTO:")
for op in sintesis.get("oportunidades_producto", []):
    print(f"  → {op}")

print("\n❓ HIPÓTESIS A VALIDAR:")
for h in sintesis.get("hipotesis_a_validar", []):
    print(f"  ? {h}")

## 2. Priorización RICE del backlog

In [ ]:
FEATURES_BACKLOG = [
    {
        "nombre": "Offboarding automático de empleados",
        "descripcion": "Al marcar a un empleado como inactivo, revocar automáticamente accesos en todos los sistemas conectados",
        "contexto": "Hay 3 incidentes de seguridad en el último año por esto. Afecta a todos los clientes enterprise."
    },
    {
        "nombre": "Dashboard de licencias sin usar",
        "descripcion": "Vista de licencias de software con uso < 1 vez/mes para identificar desperdicio",
        "contexto": "Solicitud de CFOs. 60% de cuentas enterprise tienen este tipo de rol. Diferenciador vs competencia."
    },
    {
        "nombre": "Onboarding guiado con templates de rol",
        "descripcion": "Templates predefinidos de accesos por rol (developer, sales, HR) que se aplican con un click",
        "contexto": "Problema mencionado en el 80% de entrevistas. Reduce tiempo de onboarding de 3 semanas a 1 día."
    },
    {
        "nombre": "App móvil para aprobar solicitudes",
        "descripcion": "Notificación push cuando hay una solicitud de acceso pendiente, aprobar/denegar desde el móvil",
        "contexto": "Managers dicen que aprueban tarde porque están en reuniones. Pequeño impacto en operativa."
    }
]

def puntuar_feature_rice(feature: dict) -> dict:
    """Calcula el score RICE con ayuda de IA."""
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        messages=[{
            "role": "user",
            "content": f"""Evalúa con el marco RICE:

Feature: {feature['nombre']}
Descripción: {feature['descripcion']}
Contexto: {feature['contexto']}

Base: ~200 empresas cliente, teams de 20-500 empleados.

JSON:
{{
  "reach": 150,
  "impact": 2,
  "confidence": 80,
  "effort": 4,
  "razonamiento": "breve"
}}

Impact scale: 3=masivo, 2=alto, 1=medio, 0.5=bajo, 0.25=mínimo
Effort: persona-semanas"""
        }]
    )
    texto = resp.content[0].text
    if "```" in texto:
        texto = texto.split("```")[1].lstrip("json\n")
    try:
        r = json.loads(texto)
        r["rice_score"] = round(r["reach"] * r["impact"] * r["confidence"] / 100 / r["effort"], 1)
        return r
    except (json.JSONDecodeError, ZeroDivisionError, KeyError):
        return {"rice_score": 0, "error": texto}


print("PRIORIZACIÓN RICE DEL BACKLOG")
print("=" * 60)

features_puntuadas = []
for f in FEATURES_BACKLOG:
    rice = puntuar_feature_rice(f)
    features_puntuadas.append({**f, "rice": rice})
    print(f"  {f['nombre']}: RICE={rice.get('rice_score', 0)}")

# Ordenar por RICE
features_ordenadas = sorted(features_puntuadas, key=lambda x: x["rice"].get("rice_score", 0), reverse=True)

print("\n📊 RANKING FINAL:")
for i, f in enumerate(features_ordenadas, 1):
    rice = f["rice"]
    print(f"\n  #{i} {f['nombre']}")
    print(f"      RICE: {rice.get('rice_score')} | Reach: {rice.get('reach')} | Impact: {rice.get('impact')} | Effort: {rice.get('effort')}sem")
    print(f"      Razonamiento: {rice.get('razonamiento', 'N/A')[:100]}")

## 3. Generador de user stories

In [ ]:
def generar_user_story(problema: str, perfil: str, contexto_producto: str) -> dict:
    """Genera user story completa con criterios de aceptación."""
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=900,
        messages=[{
            "role": "user",
            "content": f"""Genera una user story completa:

Problema: {problema}
Perfil usuario: {perfil}
Contexto producto: {contexto_producto}

JSON:
{{
  "titulo": "título conciso",
  "user_story": "Como [tipo], quiero [acción], para [beneficio]",
  "criterios_aceptacion": ["Dado [ctx], cuando [acción], entonces [resultado]"],
  "out_of_scope": ["qué NO incluye"],
  "metricas_exito": ["métrica con valor objetivo"],
  "estimacion": "XS/S/M/L/XL"
}}"""
        }]
    )
    texto = resp.content[0].text
    if "```" in texto:
        texto = texto.split("```")[1].lstrip("json\n")
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        return {"user_story": texto}


# Generar user stories para las top 2 features
top_2_features = features_ordenadas[:2]

for feature in top_2_features:
    story = generar_user_story(
        feature["descripcion"],
        "IT Manager / HR Admin de empresa con 50-500 empleados",
        "SaaS de gestión de accesos y licencias de software B2B"
    )

    print(f"\n{'='*60}")
    print(f"USER STORY: {story.get('titulo', feature['nombre'])}")
    print(f"{'='*60}")
    print(f"\n{story.get('user_story', '')}")
    print(f"\nEstimación: {story.get('estimacion', 'N/A')}")

    print("\n✅ Criterios de aceptación:")
    for ca in story.get("criterios_aceptacion", []):
        print(f"  • {ca}")

    print("\n📊 Métricas de éxito:")
    for m in story.get("metricas_exito", []):
        print(f"  • {m}")

## 4. Análisis de reviews de app store

In [ ]:
# Reviews simuladas (como si fueran de G2 o Capterra)
REVIEWS = [
    {"rating": 2, "texto": "El onboarding es una pesadilla. Tardamos semanas en configurar cada empleado nuevo. La interfaz es anticuada."},
    {"rating": 4, "texto": "Muy buen control de accesos. Nos ha salvado en auditorías. Echo de menos integración con Slack."},
    {"rating": 3, "texto": "Funciona pero es lento. Las APIs son buenas pero la UI deja mucho que desear. Soporte muy lento."},
    {"rating": 5, "texto": "Desde que lo implementamos hemos reducido el tiempo de onboarding de 3 semanas a 2 días. Increíble ROI."},
    {"rating": 1, "texto": "Tuvimos un problema de seguridad porque el offboarding no era automático. Cliente muy insatisfecho."},
    {"rating": 4, "texto": "La vista de licencias es fantástica, hemos ahorrado 20.000€ el primer año eliminando licencias sin usar."},
    {"rating": 3, "texto": "El precio es alto para lo que ofrece. Los competidores tienen más integraciones. El soporte es bueno."},
    {"rating": 5, "texto": "Perfecto para empresas medianas. Los templates de roles son un game-changer. Muy recomendable."},
    {"rating": 2, "texto": "No hay app móvil. Tener que aprobar accesos desde el ordenador es un problema cuando estás en reuniones."},
    {"rating": 4, "texto": "Los informes de auditoría han simplificado nuestras revisiones anuales enormemente."}
]

def analizar_reviews(reviews: list) -> dict:
    """Analiza reviews para extraer insights de producto."""
    texto_reviews = "\n".join(
        f"[{r['rating']}★] {r['texto']}"
        for r in reviews
    )
    rating_medio = sum(r["rating"] for r in reviews) / len(reviews)

    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1000,
        messages=[{
            "role": "user",
            "content": f"""Analiza estas {len(reviews)} reviews (rating medio: {rating_medio:.1f}/5):

{texto_reviews}

JSON:
{{
  "problemas_frecuentes": [{{"problema": "desc", "reviews": 3}}],
  "fortalezas": ["fortaleza mencionada"],
  "solicitudes_frecuentes": ["feature pedida"],
  "sentiment_general": "positivo/mixto/negativo",
  "acciones_prioritarias": [{{"accion": "qué hacer", "impacto": "en qué mejora"}}]
}}"""
        }]
    )
    texto = resp.content[0].text
    if "```" in texto:
        texto = texto.split("```")[1].lstrip("json\n")
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        return {"analisis": texto}


insights = analizar_reviews(REVIEWS)
rating_medio = sum(r["rating"] for r in REVIEWS) / len(REVIEWS)

print(f"ANÁLISIS DE {len(REVIEWS)} REVIEWS (Rating medio: {rating_medio:.1f}/5)")
print("=" * 60)
print(f"\nSentiment general: {insights.get('sentiment_general', 'N/A').upper()}")

print("\n❌ Problemas frecuentes:")
for p in insights.get("problemas_frecuentes", []):
    print(f"  • {p.get('problema')} ({p.get('reviews', '?')} reviews)")

print("\n✅ Fortalezas:")
for f in insights.get("fortalezas", []):
    print(f"  • {f}")

print("\n💡 Acciones prioritarias:")
for acc in insights.get("acciones_prioritarias", []):
    print(f"  → {acc.get('accion')}: {acc.get('impacto')}")

## 5. Diseño de experimento A/B

In [ ]:
def disenar_experimento_ab(hipotesis: str, metrica: str, contexto: str) -> dict:
    """Diseña un experimento A/B estructurado."""
    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=800,
        messages=[{
            "role": "user",
            "content": f"""Diseña un experimento A/B:

Hipótesis: {hipotesis}
Métrica objetivo: {metrica}
Contexto: {contexto}

JSON:
{{
  "control": "descripción del control",
  "tratamiento": "descripción del tratamiento",
  "metrica_primaria": "nombre y cómo medirla",
  "metricas_guardianes": ["métrica que no debe empeorar"],
  "tamaño_muestra": 500,
  "duracion_dias": 14,
  "criterio_exito": "qué resultado valida la hipótesis",
  "riesgos": ["riesgo del experimento"]
}}"""
        }]
    )
    texto = resp.content[0].text
    if "```" in texto:
        texto = texto.split("```")[1].lstrip("json\n")
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        return {"diseno": texto}


experimento = disenar_experimento_ab(
    hipotesis="Mostrar templates de rol en el primer paso del onboarding reducirá el tiempo de configuración inicial",
    metrica="Tiempo hasta primer empleado configurado completamente (en horas)",
    contexto="SaaS de gestión de accesos, ~50 nuevas cuentas/mes, onboarding actual dura 5-10 días de media"
)

print("DISEÑO DE EXPERIMENTO A/B")
print("=" * 60)
print(f"\n🔵 Control: {experimento.get('control', 'N/A')}")
print(f"🟢 Tratamiento: {experimento.get('tratamiento', 'N/A')}")
print(f"\n📊 Métrica primaria: {experimento.get('metrica_primaria', 'N/A')}")
print(f"👮 Métricas guardianes: {', '.join(experimento.get('metricas_guardianes', []))}")
print(f"\n👥 Tamaño de muestra: {experimento.get('tamaño_muestra', 'N/A')} cuentas")
print(f"📅 Duración: {experimento.get('duracion_dias', 'N/A')} días")
print(f"✅ Criterio de éxito: {experimento.get('criterio_exito', 'N/A')}")
print("\n⚠️ Riesgos:")
for r in experimento.get("riesgos", []):
    print(f"  • {r}")